# Theme Park Wait Time:  
## Temporal Dynamics & Optimization

### Authors  
Andrew Kim, Nolan Chu, Kyle Le, Leonardo Bangayan, Ryan Xavier  

---

## Project Overview: Analyzing New Ride Stabilization Using Statistical Inference

In this project, we will combine multiple wait time datasets from different theme parks—Disney, Universal, Six Flags, and Knott’s—to study how long it takes for newly opened rides to reach stable, long-term operating levels. The core idea is that when a new ride launches, its wait times are usually volatile due to hype, marketing, and irregular early-day operations. Over time, the ride should settle into a consistent pattern that reflects its true long-run popularity.

Using statistical inference, we will:

- Identify the opening period for each new ride.  
- Quantify when the ride transitions from volatile early wait times to statistically stable behavior.  
- Use measures such as variance, confidence intervals, and distributional changes to detect stabilization.  
- Compare stabilization timelines across parks to understand whether certain parks stabilize faster or slower and why.

Our goal is to build a data-driven methodology for detecting “time to stability” for new attractions, offering insights into theme park operations, guest behavior, and ride popularity dynamics.


In [17]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns


from datetime import datetime




In [18]:
files = {
    "disney": "../data/wait_times_2015_2025_disney.csv",
    "sixflags": "../data/wait_times_2015_2025_sixflags.csv",
    "knotts": "../data/wait_times_2015_2025_knotts.csv",
    "universal": "../data/wait_times_2015_2025_universal.csv",
    "dca": "../data/wait_times_2015_2024_DCA.csv"
}

dfs = {k: pd.read_csv(v) for k, v in files.items()}


In [19]:
for name, df in dfs.items():
    print(f"\n=== {name.upper()} ===")
    display(df.head())
    display(df.describe(include='all'))



=== DISNEY ===


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
0,2015-01-01,Space Mountain,50.0,96.0
1,2015-01-01,Indiana Jones™ Adventure,48.0,91.0
2,2015-01-01,Peter Pan's Flight,42.0,50.0
3,2015-01-01,Alice in Wonderland,37.0,50.0
4,2015-01-01,Roger Rabbit's Car Toon Spin,34.0,61.0


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
count,174682,174682,85281.000000,84902.000000
unique,3133,167,NaN,NaN
top,2015-07-04,Space Mountain,NaN,NaN
freq,94,3133,NaN,NaN
mean,NaN,NaN,21.007294,36.195025
std,NaN,NaN,14.859333,26.973308
min,NaN,NaN,0.000000,5.000000
25%,NaN,NaN,10.000000,15.000000
50%,NaN,NaN,18.000000,30.000000
75%,NaN,NaN,29.000000,50.000000



=== SIXFLAGS ===


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
0,2017-01-01,Santa's Wild Sleigh Ride,100.0,100.0
1,2017-01-01,X2,70.0,70.0
2,2017-01-01,Full Throttle,68.0,70.0
3,2017-01-01,Twisted Colossus,52.0,55.0
4,2017-01-01,Tatsu,51.0,55.0


...

,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
0,2018-07-19,TRANSFORMERS™: The Ride-3D,61.0,85.0
1,2018-07-19,Jurassic World — The Ride,56.0,80.0
2,2018-07-19,The Simpsons Ride™,49.0,60.0
3,2018-07-19,Harry Potter and the Forbidden Journey™,47.0,95.0
4,2018-07-19,Revenge of the Mummy – The Ride,47.0,65.0


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
count,30347,30347,22480.000000,22454.000000
unique,1959,66,NaN,NaN
top,2024-12-31,TRANSFORMERS™: The Ride-3D,NaN,NaN
freq,23,1959,NaN,NaN
mean,NaN,NaN,30.675089,53.061593
std,NaN,NaN,22.383210,35.372396
min,NaN,NaN,0.000000,5.000000
25%,NaN,NaN,15.000000,30.000000
50%,NaN,NaN,25.000000,45.000000
75%,NaN,NaN,40.000000,70.000000



=== DCA ===


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
0,2015-01-01,Radiator Springs Racers,82.0,166.0
1,2015-01-01,Toy Story Midway Mania!,52.0,75.0
2,2015-01-01,Pixar Pal-A-Round - Swinging,38.0,80.0
3,2015-01-01,"Monsters, Inc. Mike & Sulley to the Rescue!",28.0,60.0
4,2015-01-01,For the First Time in Forever: A Frozen Sing-A...,26.0,27.0


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
count,116743,116743,53752.000000,53539.000000
unique,3132,131,NaN,NaN
top,2015-12-22,Radiator Springs Racers,NaN,NaN
freq,66,3132,NaN,NaN
mean,NaN,NaN,21.705778,38.103887
std,NaN,NaN,20.389758,35.530112
min,NaN,NaN,0.000000,3.000000
25%,NaN,NaN,6.000000,10.000000
50%,NaN,NaN,15.000000,30.000000
75%,NaN,NaN,30.000000,55.000000


In [20]:
# unpack dictionary into separate dataframes
disney_df    = dfs["disney"].copy()
sixflags_df  = dfs["sixflags"].copy()
knotts_df    = dfs["knotts"].copy()
universal_df = dfs["universal"].copy()
dca_df       = dfs["dca"].copy()

### Initial Data Observation

Based on the preview of the data (`head()`), it is clear that not every entry in the dataset corresponds to an actual ride. Some rows reflect non-ride entities such as parades, shows, general park areas, or improperly labeled entries. Because of this, we will need to manually clean the datasets to ensure that only valid ride wait times are included in our analysis. This step is essential for producing accurate statistical results when examining ride stabilization patterns.


In [41]:
# Disneyland
print("\n=== UNIQUE RIDES: DISNEYLAND ===")
disney_rides = sorted(disney_df["Ride"].dropna().unique())
print(f"Number of unique rides: {len(disney_rides)}")
print(disney_rides)

# DCA
print("\n=== UNIQUE RIDES: DCA ===")
dca_rides = sorted(dca_df["Ride"].dropna().unique())
print(f"Number of unique rides: {len(dca_rides)}")
print(dca_rides)

# Universal
print("\n=== UNIQUE RIDES: UNIVERSAL ===")
universal_rides = sorted(universal_df["Ride"].dropna().unique())
print(f"Number of unique rides: {len(universal_rides)}")
print(universal_rides)

# Knotts
print("\n=== UNIQUE RIDES: KNOTTS ===")
knotts_rides = sorted(knotts_df["Ride"].dropna().unique())
print(f"Number of unique rides: {len(knotts_rides)}")
print(knotts_rides)

# Six Flags
print("\n=== UNIQUE RIDES: SIX FLAGS ===")
sixflags_rides = sorted(sixflags_df["Ride"].dropna().unique())
print(f"Number of unique rides: {len(sixflags_rides)}")
print(sixflags_rides)







=== UNIQUE RIDES: DISNEYLAND ===
Number of unique rides: 167
['"Ant-Man and The Wasp" Sneak Peek at Tomorrowland Theater', '"Fantasy In The Sky"*', '"Minnie & Friends - Breakfast in the Park"', '"it\'s a small world" Holiday', 'A Christmas Fantasy Parade', 'A Special 4th of July Concert Featuring The Third Marine Aircraft Wing Band', 'A Special 4th of July Concert Featuring The Third Marine Aircraft Wing Band at Big Thunder Ranch Jamboree', 'Adventureland Treehouse inspired by Walt Disney’s Swiss Family Robinson', 'Alice in Wonderland', 'Astro Orbitor', 'Autopia', 'Big Thunder Mountain Railroad', 'Big Thunder Ranch', 'Blue Bayou Restaurant', 'Buzz Lightyear Astro Blasters', 'Cafe Orleans', 'Captain America: The Living Legend and Symbol of Courage', 'Captain EO', 'Carnation Café', 'Casey Jr. Circus Train', 'Character Greetings at Mickey’s Halloween Party', "Chip 'n Dale Treehouse", "Chip 'n' Dale's GADGETcoaster", "Davy Crockett's Explorer Canoes", 'Diamond Celebration Décor', 'Disney 

### Cleaning the Disneyland Dataset

The original Disneyland dataset contained many entries that were not actual rides, such as parades, shows, character meet-and-greets, restaurants, special events, and seasonal overlays. Since our analysis focuses specifically on ride wait times, we manually filtered out all non-ride entries.

To do this, we created a curated list of all true Disneyland attractions that operate as rides and removed any rows that did not match this list. We also standardized several naming inconsistencies (e.g., holiday overlays, single-rider entries, and alternate names) to ensure that each ride is represented consistently in the dataset.

This manual cleaning step ensures that the Disneyland dataset contains only valid ride data, allowing for accurate comparison, statistical inference, and analysis of ride behavior over time.


In [30]:
# your cleaned ride list
keep_rides = [
    "Alice in Wonderland",
    "Casey Jr. Circus Train",
    "Dumbo the Flying Elephant",
    "it's a small world",
    "King Arthur Carrousel",
    "Mad Tea Party",
    "Matterhorn Bobsleds",
    "Mr. Toad's Wild Ride",
    "Peter Pan's Flight",
    "Pinocchio's Daring Journey",
    "Snow White's Enchanted Wish",
    "Storybook Land Canal Boats",
    "Astro Orbitor",
    "Autopia",
    "Buzz Lightyear Astro Blasters",
    "Finding Nemo Submarine Voyage",
    "Space Mountain",
    "Star Tours – The Adventures Continue",
    "Jungle Cruise",
    "Indiana Jones Adventure",
    "The Many Adventures of Winnie the Pooh",
    "Haunted Mansion",
    "Pirates of the Caribbean",
    "Big Thunder Mountain Railroad",
    "Davy Crockett's Explorer Canoes",
    "Mark Twain Riverboat",
    "Sailing Ship Columbia",
    "Chip 'n' Dale's Gadgetcoaster",
    "Roger Rabbit's Car Toon Spin",
    "Mickey & Minnie’s Runaway Railway",
    "Millennium Falcon: Smugglers Run",
    "Star Wars: Rise of the Resistance",
    "Tiana's Bayou Adventure",
    "Disneyland Railroad"
]

# filter df_disney to only these rides
df_disney_clean = disney_df[disney_df["Ride"].isin(keep_rides)].copy()

unique_rides = df_disney_clean["Ride"].dropna().unique()
print(f"Number of unique rides: {len(unique_rides)}")
print(sorted(unique_rides))

df_disney_clean.head()

Number of unique rides: 28
['Alice in Wonderland', 'Astro Orbitor', 'Autopia', 'Big Thunder Mountain Railroad', 'Buzz Lightyear Astro Blasters', 'Casey Jr. Circus Train', "Davy Crockett's Explorer Canoes", 'Disneyland Railroad', 'Dumbo the Flying Elephant', 'Finding Nemo Submarine Voyage', 'Jungle Cruise', 'King Arthur Carrousel', 'Mad Tea Party', 'Mark Twain Riverboat', 'Matterhorn Bobsleds', 'Millennium Falcon: Smugglers Run', "Mr. Toad's Wild Ride", "Peter Pan's Flight", "Pinocchio's Daring Journey", 'Pirates of the Caribbean', "Roger Rabbit's Car Toon Spin", 'Sailing Ship Columbia', "Snow White's Enchanted Wish", 'Space Mountain', 'Star Wars: Rise of the Resistance', 'Storybook Land Canal Boats', 'The Many Adventures of Winnie the Pooh', "Tiana's Bayou Adventure"]


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
0,2015-01-01,Space Mountain,50.0,96.0
2,2015-01-01,Peter Pan's Flight,42.0,50.0
3,2015-01-01,Alice in Wonderland,37.0,50.0
4,2015-01-01,Roger Rabbit's Car Toon Spin,34.0,61.0
5,2015-01-01,Pirates of the Caribbean,28.0,60.0


### Applying the Same Cleaning Process Across All Parks

The detailed cleaning steps we performed on the Disneyland dataset will also be applied to every other park dataset in our analysis. This includes:

- **Identifying the true ride attractions** for each park  
- **Removing all non-ride entries**, such as shows, parades, character experiences, dining locations, shops, and seasonal events  
- **Standardizing naming inconsistencies**, including punctuation differences, alternate spellings, overlays, and old ride names  
- **Dropping single-rider listings** unless explicitly included in the core ride set  
- Ensuring that each dataset contains **only real rides**, with consistent naming, ready for statistical analysis  

By applying the same methodology uniformly across Disneyland, Disney California Adventure, Universal Studios, Knott’s Berry Farm, and Six Flags, we ensure that each dataset is comparable, reliable, and suitable for our broader study of ride wait-time stabilization dynamics.


In [38]:
dca_rides = [
    "Radiator Springs Racers",
    "Incredicoaster",
    "Guardians of the Galaxy – Mission: BREAKOUT!",
    "Toy Story Midway Mania!",
    "Pixar Pal-A-Round",
    "Jessie’s Critter Carousel",
    "Inside Out Emotional Whirlwind",
    "Luigi's Rollickin' Roadsters",
    "Mater's Junkyard Jamboree",
    "The Little Mermaid – Ariel’s Undersea Adventure",
    "Golden Zephyr",
    "Goofy's Sky School",
    "Jumpin' Jellyfish",
    "Silly Symphony Swings",
    "Grizzly River Run",
    "Soarin' Around the World",
    "WEB SLINGERS: A Spider-Man Adventure"
]

dca_map = {
    # Guardians variants
    "Guardians of the Galaxy - Mission: BREAKOUT!": "Guardians of the Galaxy – Mission: BREAKOUT!",
    "Guardians of the Galaxy - Monsters After Dark": "Guardians of the Galaxy – Mission: BREAKOUT!",
    "Guardians of the Galaxy – Monsters After Dark": "Guardians of the Galaxy – Mission: BREAKOUT!",

    # Jessie apostrophe variant
    "Jessie's Critter Carousel": "Jessie’s Critter Carousel",

    # Luigi overlays / old names -> treat as same ride
    "Luigi's Flying Tires, presented by Alamo": "Luigi's Rollickin' Roadsters",
    "Luigi's Honkin' Haul-O-Ween": "Luigi's Rollickin' Roadsters",
    "Luigi's Joy to the Whirl": "Luigi's Rollickin' Roadsters",

    # Pixar Pal-A-Round variants
    "Pixar Pal-A-Round – Non-Swinging": "Pixar Pal-A-Round",
    "Pixar Pal-A-Round - Swinging": "Pixar Pal-A-Round",

    # Mermaid naming variant
    "The Little Mermaid - Ariel's Undersea Adventure": "The Little Mermaid – Ariel’s Undersea Adventure",

    # Soarin punctuation variant
    "Soarin’ Around the World": "Soarin' Around the World"
}

dca_df = dca_df.copy()
dca_df["Ride_clean"] = dca_df["Ride"].replace(dca_map)

# Keep ONLY the official rides (no single rider updates)
dca_df = dca_df[dca_df["Ride_clean"].isin(dca_rides)].copy()

# Finalize
dca_df["Ride"] = dca_df["Ride_clean"]
dca_df = dca_df.drop(columns=["Ride_clean"])

sorted(dca_df["Ride"].unique())


['Golden Zephyr',
 "Goofy's Sky School",
 'Grizzly River Run',
 'Guardians of the Galaxy – Mission: BREAKOUT!',
 'Incredicoaster',
 'Inside Out Emotional Whirlwind',
 'Jessie’s Critter Carousel',
 "Jumpin' Jellyfish",
 'Pixar Pal-A-Round',
 'Radiator Springs Racers',
 'Silly Symphony Swings',
 "Soarin' Around the World",
 'Toy Story Midway Mania!',
 'WEB SLINGERS: A Spider-Man Adventure']

In [39]:
ush_rides = [
    "Studio Tour",
    "Mario Kart: Bowser’s Challenge",
    "The Secret Life of Pets: Off the Leash",
    "Jurassic World – The Ride",
    "Revenge of the Mummy – The Ride",
    "TRANSFORMERS: The Ride 3D",
    "Harry Potter and the Forbidden Journey",
    "Flight of the Hippogriff",
    "Despicable Me: Minion Mayhem",
    "The Simpsons Ride",
    "Kung Fu Panda Adventure"
]

ush_map = {
    "Despicable Me Minion Mayhem": "Despicable Me: Minion Mayhem",
    "Flight of the Hippogriff™": "Flight of the Hippogriff",
    "Harry Potter and the Forbidden Journey™": "Harry Potter and the Forbidden Journey",
    "Jurassic World\xa0— The Ride": "Jurassic World – The Ride",
    "Mario Kart™: Bowser’s Challenge": "Mario Kart: Bowser’s Challenge",
    "TRANSFORMERS™: The Ride-3D": "TRANSFORMERS: The Ride 3D",
    "The Simpsons Ride™": "The Simpsons Ride"
}

universal_df = universal_df.copy()
universal_df["Ride_clean"] = universal_df["Ride"].replace(ush_map)

universal_df = universal_df[universal_df["Ride_clean"].isin(ush_rides)].copy()

universal_df["Ride"] = universal_df["Ride_clean"]
universal_df = universal_df.drop(columns=["Ride_clean"])

sorted(universal_df["Ride"].unique())


['Despicable Me: Minion Mayhem',
 'Flight of the Hippogriff',
 'Harry Potter and the Forbidden Journey',
 'Jurassic World – The Ride',
 'Kung Fu Panda Adventure',
 'Mario Kart: Bowser’s Challenge',
 'Revenge of the Mummy – The Ride',
 'Studio Tour',
 'TRANSFORMERS: The Ride 3D',
 'The Secret Life of Pets: Off the Leash',
 'The Simpsons Ride']

In [ ]:
knotts_rides = [
    "GhostRider",
    "HangTime",
    "Silver Bullet",
    "Xcelerator",
    "Supreme Scream",
    "Jaguar!",
    "Pony Express",
    "Sierra Sidewinder",
    "Timber Mountain Log Ride",
    "Calico Mine Ride",
    "Calico River Rapids",
    "Sol Spin",
    "Coast Rider",
    "Beagle Express Railroad",
    "Balloon Race"
]

knotts_map = {
    # fixes in the dataset
    "Calico Candy Mine Ride": "Calico Mine Ride",
    "Grand Sierra Railroad": "Beagle Express Railroad",
    "Grand Sierra R.R.": "Beagle Express Railroad",
    "Beagle Express": "Beagle Express Railroad",
    
    # Timber Mountain variants
    "Timber Mountain Log Ride: Halloween Hootenanny": "Timber Mountain Log Ride",
    
    # Xcelerator naming
    "Xcelerator The Ride": "Xcelerator"
}

knotts_df = knotts_df.copy()
knotts_df["Ride_clean"] = knotts_df["Ride"].replace(knotts_map)

# keep only the real rides
knotts_df = knotts_df[knotts_df["Ride_clean"].isin(knotts_rides)].copy()

# finalize
knotts_df["Ride"] = knotts_df["Ride_clean"]
knotts_df = knotts_df.drop(columns=["Ride_clean"])

sorted(knotts_df["Ride"].unique())


['Balloon Race',
 'Beagle Express Railroad',
 'Calico Mine Ride',
 'Calico River Rapids',
 'Coast Rider',
 'GhostRider',
 'HangTime',
 'Jaguar!',
 'Pony Express',
 'Sierra Sidewinder',
 'Silver Bullet',
 'Sol Spin',
 'Supreme Scream',
 'Timber Mountain Log Ride',
 'Xcelerator']

In [42]:
sfmm_rides = [
    "Apocalypse: The Ride",
    "Twisted Colossus",
    "Full Throttle",
    "Wonder Woman: Flight of Courage",
    "BATMAN: The Ride",
    "Canyon Blaster",
    "Gold Rusher",
    "Goliath",
    "Ninja",
    "Jet Stream",
    "The Riddler's Revenge",
    "Scream!",
    "Road Runner Express",
    "Grand American Carousel",
    "Buccaneer",
    "Swashbuckler",
    "Castaway Cove",
    "Magic Flyer",
    "Whistlestop Train",
    "Bonzai Pipelines"
]

sfmm_map = {
    # Name variants in the DF
    "Apocalypse": "Apocalypse: The Ride",
    "BATMAN The Ride": "BATMAN: The Ride",
    "Scream": "Scream!",
    "THE RIDDLER’S Revenge": "The Riddler's Revenge",
    "WONDER WOMAN™ Flight of Courage": "Wonder Woman: Flight of Courage"
    # note: WONDER WOMAN Lasso of Truth is a different ride -> dropped
}

sixflags_df = sixflags_df.copy()
sixflags_df["Ride_clean"] = sixflags_df["Ride"].replace(sfmm_map)

sixflags_df = sixflags_df[sixflags_df["Ride_clean"].isin(sfmm_rides)].copy()

sixflags_df["Ride"] = sixflags_df["Ride_clean"]
sixflags_df = sixflags_df.drop(columns=["Ride_clean"])

sorted(sixflags_df["Ride"].unique())


['Apocalypse: The Ride',
 'BATMAN: The Ride',
 'Buccaneer',
 'Canyon Blaster',
 'Full Throttle',
 'Gold Rusher',
 'Goliath',
 'Grand American Carousel',
 'Jet Stream',
 'Magic Flyer',
 'Ninja',
 'Road Runner Express',
 'Scream!',
 'Swashbuckler',
 "The Riddler's Revenge",
 'Twisted Colossus',
 'Whistlestop Train',
 'Wonder Woman: Flight of Courage']

### Returning to Exploratory Data Analysis (EDA)

Now that all five park datasets have been fully cleaned and standardized—ensuring only true ride entries remain and all naming inconsistencies have been resolved—we can move back into exploratory data analysis (EDA). With the noise removed, our summaries and visualizations will more accurately reflect actual ride behavior.

In this next phase, we will examine key descriptive statistics such as minimums, maximums, and average wait times, allowing us to compare ride performance across parks and identify meaningful patterns. This cleaned foundation sets us up for deeper statistical inference and more reliable insights into ride dynamics and stabilization trends.


In [44]:
cols = ["Average Wait Time (mins)", "Max Wait Time (mins)"]

dfs_to_check = {
    "Disneyland": disney_df,
    "DCA": dca_df,
    "Universal": universal_df,
    "Knott's": knotts_df,
    "Six Flags": sixflags_df
}

for park, df in dfs_to_check.items():
    print(f"\n===== {park.upper()} =====")
    display(df[cols].describe())



===== DISNEYLAND =====


,Average Wait Time (mins),Max Wait Time (mins)
count,85281.000000,84902.000000
mean,21.007294,36.195025
std,14.859333,26.973308
min,0.000000,5.000000
25%,10.000000,15.000000
50%,18.000000,30.000000
75%,29.000000,50.000000
max,169.000000,900.000000



===== DCA =====


,Average Wait Time (mins),Max Wait Time (mins)
count,29444.000000,29357.000000
mean,27.636632,48.630412
std,23.870223,41.760294
min,0.000000,4.000000
25%,7.000000,10.000000
50%,20.000000,45.000000
75%,42.000000,75.000000
max,185.000000,900.000000



===== UNIVERSAL =====


,Average Wait Time (mins),Max Wait Time (mins)
count,18621.000000,18601.000000
mean,32.890876,57.055051
std,22.571015,36.280886
min,0.000000,5.000000
25%,16.000000,30.000000
50%,28.000000,50.000000
75%,43.000000,75.000000
max,193.000000,300.000000



===== KNOTT'S =====


,Average Wait Time (mins),Max Wait Time (mins)
count,20400.000000,19766.000000
mean,33.487647,48.016594
std,30.555443,41.763763
min,0.000000,1.000000
25%,9.000000,15.000000
50%,25.000000,40.000000
75%,47.000000,65.000000
max,240.000000,450.000000



===== SIX FLAGS =====


,Average Wait Time (mins),Max Wait Time (mins)
count,35287.000000,33530.000000
mean,19.684048,27.171608
std,25.278484,32.648637
min,0.000000,1.000000
25%,5.000000,5.000000
50%,8.000000,10.000000
75%,26.000000,45.000000
max,220.000000,450.000000


### Investigating Zero Wait Times

During our exploratory analysis, we noticed that several entries across the datasets reported a **0-minute wait time**. This is suspicious, especially for Disneyland and Disney California Adventure, because these parks **never publish a true zero wait time**. Even in the least crowded moments, Disney’s officially reported wait times default to **5 minutes** rather than zero.

Because of this, any zero values in our datasets likely indicate:

- **Missing or incomplete data**
- **System logging errors**
- **Temporary ride closures or downtime recorded incorrectly**
- **Scraping or API inconsistencies during data collection**

Identifying these anomalies is important, as zero wait times can distort averages, minima, and overall ride behavior. In later cleaning steps, we may need to remove or adjust these zero values to ensure the integrity of our statistical analysis.


In [45]:
cols = ["Average Wait Time (mins)", "Max Wait Time (mins)"]

dfs_to_check = {
    "Disneyland": disney_df,
    "DCA": dca_df,
    "Universal": universal_df,
    "Knott's": knotts_df,
    "Six Flags": sixflags_df
}

for park, df in dfs_to_check.items():
    print(f"\n===== {park.upper()} =====")
    for c in cols:
        zero_count = (df[c] == 0).sum()
        print(f"{c}: {zero_count} zeros")



===== DISNEYLAND =====
Average Wait Time (mins): 348 zeros
Max Wait Time (mins): 0 zeros

===== DCA =====
Average Wait Time (mins): 83 zeros
Max Wait Time (mins): 0 zeros

===== UNIVERSAL =====
Average Wait Time (mins): 10 zeros
Max Wait Time (mins): 0 zeros

===== KNOTT'S =====
Average Wait Time (mins): 179 zeros
Max Wait Time (mins): 0 zeros

===== SIX FLAGS =====
Average Wait Time (mins): 387 zeros
Max Wait Time (mins): 0 zeros


In [46]:
cols = ["Average Wait Time (mins)", "Max Wait Time (mins)"]

dfs_to_check = {
    "Disneyland": disney_df,
    "DCA": dca_df,
    "Universal": universal_df,
    "Knott's": knotts_df,
    "Six Flags": sixflags_df
}

for park, df in dfs_to_check.items():
    print(f"\n===== FIRST 5 ZERO-WAIT ROWS: {park.upper()} =====")
    
    zero_rows = df[(df["Average Wait Time (mins)"] == 0) | 
                   (df["Max Wait Time (mins)"] == 0)]
    
    display(zero_rows.head())



===== FIRST 5 ZERO-WAIT ROWS: DISNEYLAND =====


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
42743,2017-02-15,"""it's a small world"" Holiday",0.0,NaN
42744,2017-02-15,Alice in Wonderland,0.0,NaN
42745,2017-02-15,Astro Orbitor,0.0,NaN
42746,2017-02-15,Autopia,0.0,NaN
42747,2017-02-15,Big Thunder Mountain Railroad,0.0,NaN



===== FIRST 5 ZERO-WAIT ROWS: DCA =====


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
88782,2022-11-16,Golden Zephyr,0.0,NaN
88783,2022-11-16,Goofy's Sky School,0.0,NaN
88784,2022-11-16,Grizzly River Run,0.0,NaN
88785,2022-11-16,Guardians of the Galaxy – Mission: BREAKOUT!,0.0,NaN
88786,2022-11-16,Incredicoaster,0.0,NaN



===== FIRST 5 ZERO-WAIT ROWS: UNIVERSAL =====


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
8862,2021-04-20,Despicable Me: Minion Mayhem,0.0,NaN
8863,2021-04-20,Flight of the Hippogriff,0.0,NaN
8864,2021-04-20,Harry Potter and the Forbidden Journey,0.0,NaN
8865,2021-04-20,Jurassic World – The Ride,0.0,NaN
8866,2021-04-20,Kung Fu Panda Adventure,0.0,NaN



===== FIRST 5 ZERO-WAIT ROWS: KNOTT'S =====


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
941,2018-08-28,Sierra Sidewinder,0.0,NaN
2439,2018-11-06,Calico Mine Ride,0.0,NaN
2440,2018-11-06,Calico River Rapids,0.0,NaN
2536,2018-11-13,Calico Mine Ride,0.0,NaN
2537,2018-11-13,Calico River Rapids,0.0,NaN



===== FIRST 5 ZERO-WAIT ROWS: SIX FLAGS =====


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
232,2017-01-05,Apocalypse: The Ride,0.0,NaN
453,2017-01-15,Apocalypse: The Ride,0.0,NaN
456,2017-01-15,The Riddler's Revenge,0.0,NaN
583,2017-01-22,Apocalypse: The Ride,0.0,NaN
584,2017-01-22,Full Throttle,0.0,NaN


### Removing Rows with Missing or Invalid Wait Time Data

While examining the rows that contain `0` for the average wait time, we observed a consistent pattern:  
these same rows also have `NaN` values in the *Max Wait Time* column. Since Disney and DCA never report a true zero wait time, and the presence of `NaN` suggests incomplete or corrupted records, these entries are unlikely to reflect real operational conditions.

To maintain data integrity and ensure our statistical analysis is based on valid observations, we will **remove all rows that contain any `NaN` values** in the wait time columns. This step prevents missing or faulty data from skewing results, especially when calculating descriptive statistics and identifying stabilization patterns.


In [47]:
cleaned_dfs = {}

for name, df in {
    "Disneyland": disney_df,
    "DCA": dca_df,
    "Universal": universal_df,
    "Knott's": knotts_df,
    "Six Flags": sixflags_df
}.items():
    cleaned_dfs[name] = df.dropna().copy()
    print(f"{name}: {len(df) - len(cleaned_dfs[name])} rows dropped due to NaN values")

# unpack cleaned dfs back into variables
disney_df    = cleaned_dfs["Disneyland"]
dca_df       = cleaned_dfs["DCA"]
universal_df = cleaned_dfs["Universal"]
knotts_df    = cleaned_dfs["Knott's"]
sixflags_df  = cleaned_dfs["Six Flags"]


Disneyland: 89780 rows dropped due to NaN values
DCA: 9823 rows dropped due to NaN values
Universal: 812 rows dropped due to NaN values
Knott's: 5515 rows dropped due to NaN values
Six Flags: 6825 rows dropped due to NaN values


In [48]:
cols = ["Average Wait Time (mins)", "Max Wait Time (mins)"]

dfs_to_check = {
    "Disneyland": disney_df,
    "DCA": dca_df,
    "Universal": universal_df,
    "Knott's": knotts_df,
    "Six Flags": sixflags_df
}

for park, df in dfs_to_check.items():
    print(f"\n===== ZERO-WAIT SUMMARY: {park.upper()} =====")
    
    # count zero values
    zero_avg = (df["Average Wait Time (mins)"] == 0).sum()
    zero_max = (df["Max Wait Time (mins)"] == 0).sum()
    print(f"Average Wait Time zeros: {zero_avg}")
    print(f"Max Wait Time zeros: {zero_max}")
    
    # show first 5 rows with zeros
    zero_rows = df[(df["Average Wait Time (mins)"] == 0) |
                   (df["Max Wait Time (mins)"] == 0)]
    
    print("\nFirst 5 rows with zeros:")
    display(zero_rows.head())



===== ZERO-WAIT SUMMARY: DISNEYLAND =====
Average Wait Time zeros: 0
Max Wait Time zeros: 0

First 5 rows with zeros:


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)



===== ZERO-WAIT SUMMARY: DCA =====
Average Wait Time zeros: 0
Max Wait Time zeros: 0

First 5 rows with zeros:


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)



===== ZERO-WAIT SUMMARY: UNIVERSAL =====
Average Wait Time zeros: 0
Max Wait Time zeros: 0

First 5 rows with zeros:


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)



===== ZERO-WAIT SUMMARY: KNOTT'S =====
Average Wait Time zeros: 0
Max Wait Time zeros: 0

First 5 rows with zeros:


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)



===== ZERO-WAIT SUMMARY: SIX FLAGS =====
Average Wait Time zeros: 0
Max Wait Time zeros: 0

First 5 rows with zeros:


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)


In [50]:
cleaned_dfs = {}

for name, df in {
    "Disneyland": disney_df,
    "DCA": dca_df,
    "Universal": universal_df,
    "Knott's": knotts_df,
    "Six Flags": sixflags_df
}.items():
    
    cleaned = df[(df["Average Wait Time (mins)"] > 0) &
                 (df["Max Wait Time (mins)"] > 0)].copy()
    
    print(f"{name}: removed {len(df) - len(cleaned)} zero-wait rows")
    
    cleaned_dfs[name] = cleaned

# unpack
disney_df    = cleaned_dfs["Disneyland"]
dca_df       = cleaned_dfs["DCA"]
universal_df = cleaned_dfs["Universal"]
knotts_df    = cleaned_dfs["Knott's"]
sixflags_df  = cleaned_dfs["Six Flags"]

print("===== DISNEYLAND =====")
display(disney_df.head())

print("===== DCA =====")
display(dca_df.head())

print("===== UNIVERSAL =====")
display(universal_df.head())

print("===== KNOTT'S =====")
display(knotts_df.head())

print("===== SIX FLAGS =====")
display(sixflags_df.head())



Disneyland: removed 0 zero-wait rows
DCA: removed 0 zero-wait rows
Universal: removed 0 zero-wait rows
Knott's: removed 0 zero-wait rows
Six Flags: removed 0 zero-wait rows
===== DISNEYLAND =====


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
0,2015-01-01,Space Mountain,50.0,96.0
1,2015-01-01,Indiana Jones™ Adventure,48.0,91.0
2,2015-01-01,Peter Pan's Flight,42.0,50.0
3,2015-01-01,Alice in Wonderland,37.0,50.0
4,2015-01-01,Roger Rabbit's Car Toon Spin,34.0,61.0


===== DCA =====


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
0,2015-01-01,Radiator Springs Racers,82.0,166.0
1,2015-01-01,Toy Story Midway Mania!,52.0,75.0
5,2015-01-01,Soarin' Around the World,26.0,53.0
6,2015-01-01,Guardians of the Galaxy – Mission: BREAKOUT!,26.0,66.0
7,2015-01-01,Incredicoaster,24.0,38.0


===== UNIVERSAL =====


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
0,2018-07-19,TRANSFORMERS: The Ride 3D,61.0,85.0
1,2018-07-19,Jurassic World – The Ride,56.0,80.0
2,2018-07-19,The Simpsons Ride,49.0,60.0
3,2018-07-19,Harry Potter and the Forbidden Journey,47.0,95.0
4,2018-07-19,Revenge of the Mummy – The Ride,47.0,65.0


===== KNOTT'S =====


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
0,2018-07-15,GhostRider,73.0,90.0
1,2018-07-15,HangTime,37.0,55.0
2,2018-07-15,Timber Mountain Log Ride,31.0,40.0
3,2018-07-15,Xcelerator,30.0,40.0
4,2018-07-15,Calico River Rapids,29.0,35.0


===== SIX FLAGS =====


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
2,2017-01-01,Full Throttle,68.0,70.0
3,2017-01-01,Twisted Colossus,52.0,55.0
5,2017-01-01,Ninja,33.0,35.0
6,2017-01-01,BATMAN: The Ride,19.0,20.0
7,2017-01-01,The Riddler's Revenge,18.0,20.0


### Transitioning from Minimum to Maximum Wait Time Analysis

After cleaning the dataset by removing rows with missing values and addressing entries that reported a zero wait time—an unrealistic scenario for parks like Disneyland and DCA—we now shift our attention to the opposite end of the spectrum: **maximum recorded wait times**.

Just as zero-minute waits can indicate faulty data collection or logging inconsistencies, excessively high wait times may also signal errors such as system glitches, ride closures, or malformed entries in the historical dataset. Before performing statistical inference or modeling ride stabilization behavior, it is crucial to verify that the upper bound of our wait time data reflects realistic park operations.

In the next step, we will inspect the maximum wait time values across all parks and identify any anomalous outliers (for example, entries reporting wait times of 900 minutes or more). These extreme values must be reviewed and potentially removed to maintain data integrity for the analyses that follow.


In [51]:
dfs_to_check = {
    "Disneyland": disney_df,
    "DCA": dca_df,
    "Universal": universal_df,
    "Knott's": knotts_df,
    "Six Flags": sixflags_df
}

for park, df in dfs_to_check.items():
    print(f"\n===== {park.upper()} =====")
    print("Max of Average Wait Time:", df["Average Wait Time (mins)"].max())
    print("Max of Max Wait Time:", df["Max Wait Time (mins)"].max())



===== DISNEYLAND =====
Max of Average Wait Time: 169.0
Max of Max Wait Time: 900.0

===== DCA =====
Max of Average Wait Time: 185.0
Max of Max Wait Time: 900.0

===== UNIVERSAL =====
Max of Average Wait Time: 193.0
Max of Max Wait Time: 300.0

===== KNOTT'S =====
Max of Average Wait Time: 240.0
Max of Max Wait Time: 450.0

===== SIX FLAGS =====
Max of Average Wait Time: 220.0
Max of Max Wait Time: 450.0


### Investigating Unrealistic Maximum Wait Times

After examining the maximum wait times across all five parks, a clear pattern emerges: several parks report extremely high values that are not operationally realistic. For instance, Disneyland and Disney California Adventure both show a maximum wait time of **900 minutes**, while Knott’s Berry Farm and Six Flags Magic Mountain share a maximum of **450 minutes**. Universal Studios Hollywood, on the other hand, tops out at a much more reasonable **300 minutes**.

These repeated maximum values strongly suggest that certain parks use **embedded default values** in their data systems. Rather than reflecting true guest wait times, these extreme numbers may indicate:

- A **system-defined placeholder** when a ride is closed or unreportable  
- A **scraper or API fallback** when no updated wait time is available  
- An internal **“error” or “override” code** expressed numerically  
- A **vendor/company-wide default** (e.g., Disney using 900, Cedar Fair & Six Flags using 450)

Supporting this idea, parks under the same ownership groups show identical upper limits:

- **Disneyland + DCA → 900 minutes**  
- **Knott’s + Six Flags → 450 minutes**  
- **Universal → 300 minutes**  

This strongly implies that these values are not true wait times but rather **default flags inserted into the dataset**.

Because these defaults dramatically distort averages, variance, and downstream statistical inference, they need to be identified and removed before modeling ride stabilization behavior. Our next step will be to filter out these unrealistic maximums and ensure that the remaining dataset reflects genuine park operations.


In [53]:
# define per-park default max wait values
default_max = {
    "Disneyland": 900,
    "DCA": 900,
    "Universal": 300,
    "Knott's": 450,
    "Six Flags": 450
}

dfs = {
    "Disneyland": disney_df,
    "DCA": dca_df,
    "Universal": universal_df,
    "Knott's": knotts_df,
    "Six Flags": sixflags_df
}

cleaned_max_dfs = {}

for park, df in dfs.items():
    max_flag = default_max[park]

    # remove rows with system-default max wait time
    cleaned = df[df["Max Wait Time (mins)"] != max_flag].copy()
    cleaned_max_dfs[park] = cleaned

    # sort by Max Wait Time descending
    sorted_df = cleaned.sort_values("Max Wait Time (mins)", ascending=False)

    print(f"\n===== {park.upper()} =====")
    print(f"Removed rows with Max Wait Time = {max_flag}: {len(df) - len(cleaned)} rows")
    print("Top rows after cleaning (sorted by Max Wait Time):")
    display(sorted_df.head())

# unpack back to your working variables
disney_df    = cleaned_max_dfs["Disneyland"]
dca_df       = cleaned_max_dfs["DCA"]
universal_df = cleaned_max_dfs["Universal"]
knotts_df    = cleaned_max_dfs["Knott's"]
sixflags_df  = cleaned_max_dfs["Six Flags"]



===== DISNEYLAND =====
Removed rows with Max Wait Time = 900: 0 rows
Top rows after cleaning (sorted by Max Wait Time):


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
81754,2018-11-30,Big Thunder Mountain Railroad,43.0,600.0
71218,2018-06-13,Space Mountain,65.0,555.0
62264,2018-01-16,Buzz Lightyear Astro Blasters,29.0,540.0
55370,2017-09-22,Indiana Jones™ Adventure,29.0,500.0
1095,2015-02-09,Star Tours - The Adventures Continue,39.0,500.0



===== DCA =====
Removed rows with Max Wait Time = 900: 0 rows
Top rows after cleaning (sorted by Max Wait Time):


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
44740,2018-03-29,Radiator Springs Racers,117.0,750.0
10661,2015-08-19,Guardians of the Galaxy – Mission: BREAKOUT!,53.0,700.0
48844,2018-07-10,Goofy's Sky School,34.0,445.0
19425,2016-05-21,Incredicoaster,34.0,400.0
10084,2015-08-10,Incredicoaster,27.0,330.0



===== UNIVERSAL =====
Removed rows with Max Wait Time = 300: 0 rows
Top rows after cleaning (sorted by Max Wait Time):


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
4236,2019-04-27,TRANSFORMERS: The Ride 3D,66.0,255.0
15317,2022-06-11,Jurassic World – The Ride,69.0,240.0
2696,2018-12-24,Harry Potter and the Forbidden Journey,161.0,240.0
2886,2019-01-03,Harry Potter and the Forbidden Journey,147.0,240.0
14680,2022-04-23,Jurassic World – The Ride,108.0,240.0



===== KNOTT'S =====
Removed rows with Max Wait Time = 450: 0 rows
Top rows after cleaning (sorted by Max Wait Time):


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
21918,2022-05-15,Coast Rider,46.0,335.0
46545,2024-03-13,HangTime,23.0,305.0
49290,2024-05-31,GhostRider,185.0,300.0
49463,2024-06-04,Xcelerator,115.0,300.0
49548,2024-06-06,GhostRider,171.0,300.0



===== SIX FLAGS =====
Removed rows with Max Wait Time = 450: 0 rows
Top rows after cleaning (sorted by Max Wait Time):


,Date,Ride,Average Wait Time (mins),Max Wait Time (mins)
108720,2024-07-06,Full Throttle,67.0,250.0
100012,2023-10-14,Twisted Colossus,154.0,240.0
89541,2022-12-28,Wonder Woman: Flight of Courage,166.0,240.0
114540,2024-12-28,Wonder Woman: Flight of Courage,105.0,240.0
114541,2024-12-28,Twisted Colossus,98.0,230.0
